In [1]:
import os 
from dotenv import load_dotenv
load_dotenv(override = True)

True

In [2]:
if os.environ['OPENAI_API_KEY']:
    print("OPENAI_API_KEY is set")

OPENAI_API_KEY is set


In [3]:
from langchain_openai import ChatOpenAI

In [4]:
llm = ChatOpenAI(model="gpt-5-nano", temperature=0)

In [5]:
response = llm.invoke('Hello, how are you?')
response.content

'Hi there! I’m doing well, thanks for asking. How can I help you today?'

### RAG implemetataion with own data ###

### STEP 1 : Preparing data for text ###

In [6]:
from langchain_core.documents import Document

my_text = """Kratos is a powerful warlord with super human strength from the land of Sparta in Greece during a time of gods and conflict. Kratos also has a family, a wife and a daughter, whom he loves very much. During a great battle, he is forced to sell his soul to the God of War Ares in order to save himself and defeat his opponent. Ares then enslaves him to do his bidding, leading him to inadvertently murder his own family. Swearing vengeance, Kratos strives to open Pandora's Box which was guarded by the titan Cronos, the enslaved father of Zeus, to obtain the power to kill Ares. He succeeds and becomes the new God of War.

His anger has not subsided though, and he rampages through Greece with his armies. Zeus, lord of the Pantheon and leader of the Greek Gods as well as revealed to be the father of Kratos, kills Kratos to end his menace, to which Kratos swears revenge. Kratos escapes hell and travels back in time by defeating and taking advantage of the Sisters of Fate who had pre-determined Kratos' destiny to die at the hands of Zeus in order to undo the events of his death. Kratos then battles his father and defeats him, but Zeus escapes.

Zeus retreats to the Pantheon and Kratos awakens the titans of old (celestials who gave birth to the gods and were later usurped by said gods, namely Zeus) with the help of the mother of all titans, Gaia, to try and destory the Pantheon. Though victorious in a battle to the death against Posiedon, Kratos and the titans are ultimately thwarted by Zeus, during which Gaia betrays Kratos in an attempt to salvage the battle. Kratos once again works his way back from hell all the way to the Pantheon, vengfully killing many other gods who attempt to stop him, as well as many titans including both Gaia and Cronos, and finally reaches Zeus once again. He kills his father after a long battle during which Athena sacrificed herself in an attempt to save Zeus.

Kratos mournes Athena, who was his ally as much as she was to Zues, and with his revenge so horribly complete, Kratos bears witness to the destruction that killing the gods has wrought on Greece from the ruins of the Pantheon as the fires of his vendetta span across the horizon. Feeling intense guilt and remourse at the sight of his selfish deeds, he releases the power he obtained from Pandora's Box, revealed to be essence of hope, to give humanity a chance to rebuild what is left Greece, now free from the influence of tyrannical gods. Kratos then seemingly dies from the wounds he sustained in the battle with Zeus.

It is revealed however that Kratos survived his injuries and traveled north. He meets a female warrior names Laufey, who Kratos endearingly refers to as Faye. They become lovers and bare a son, Atreus, who Kratos keeps his godhood a secret from to spare him both the truth of his father and the burdens of godhood. Kratos chooses to live as a mortal and let go of his former life to be the husband and father he regrets he wasn't for his previous family. At some point, Faye passes away due to unknown circumstances. Her dying wish was for Kratos and Atreus to spread her ashes atop the highest peak in all the realms (All nine realms of Norse mythology, Midgard, Alfheim, Jotunheim, Nifilheim, Asgard, Svartlheim, Musphelheim, Vanaheim, and Helheim). A perilous journey is set in motion in the unfamiliar lands of Midgard, a place ruled by the cunniving Odin and the Aesir gods, a journey which Kratos does not believe his sickly son nor he himself are prepared for without his late lover's guidance."""

docs = [Document(page_content=my_text, metadata={"source": "ABC XYZ", "documentID": "001"})]

docs

[Document(metadata={'source': 'ABC XYZ', 'documentID': '001'}, page_content="Kratos is a powerful warlord with super human strength from the land of Sparta in Greece during a time of gods and conflict. Kratos also has a family, a wife and a daughter, whom he loves very much. During a great battle, he is forced to sell his soul to the God of War Ares in order to save himself and defeat his opponent. Ares then enslaves him to do his bidding, leading him to inadvertently murder his own family. Swearing vengeance, Kratos strives to open Pandora's Box which was guarded by the titan Cronos, the enslaved father of Zeus, to obtain the power to kill Ares. He succeeds and becomes the new God of War.\n\nHis anger has not subsided though, and he rampages through Greece with his armies. Zeus, lord of the Pantheon and leader of the Greek Gods as well as revealed to be the father of Kratos, kills Kratos to end his menace, to which Kratos swears revenge. Kratos escapes hell and travels back in time by

### STEP 2 : Chunking ###

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50)

chunks = text_splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'ABC XYZ', 'documentID': '001'}, page_content='Kratos is a powerful warlord with super human strength from the land of Sparta in Greece during a time of gods and conflict. Kratos also has a family, a wife and a daughter, whom he loves very much.'),
 Document(metadata={'source': 'ABC XYZ', 'documentID': '001'}, page_content='a wife and a daughter, whom he loves very much. During a great battle, he is forced to sell his soul to the God of War Ares in order to save himself and defeat his opponent. Ares then enslaves him to'),
 Document(metadata={'source': 'ABC XYZ', 'documentID': '001'}, page_content="defeat his opponent. Ares then enslaves him to do his bidding, leading him to inadvertently murder his own family. Swearing vengeance, Kratos strives to open Pandora's Box which was guarded by the"),
 Document(metadata={'source': 'ABC XYZ', 'documentID': '001'}, page_content="to open Pandora's Box which was guarded by the titan Cronos, the enslaved father of Zeu

### STEP 3: Embedding model ###

In [8]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")


### STEP 4: Storage ###

In [11]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = embedding_model
)

### STEP 5: Semantic Search ###

In [12]:
vectorstore.similarity_search("Who is Kratos?", k=2)

[Document(metadata={'source': 'ABC XYZ', 'documentID': '001'}, page_content='Kratos is a powerful warlord with super human strength from the land of Sparta in Greece during a time of gods and conflict. Kratos also has a family, a wife and a daughter, whom he loves very much.'),
 Document(metadata={'documentID': '001', 'source': 'ABC XYZ'}, page_content='revealed to be the father of Kratos, kills Kratos to end his menace, to which Kratos swears revenge. Kratos escapes hell and travels back in time by defeating and taking advantage of the Sisters of')]

## Talk to LLM ##

In [21]:
context = vectorstore.similarity_search("Who is Kratos?", k=2)

display(context)

[Document(metadata={'documentID': '001', 'source': 'ABC XYZ'}, page_content='Kratos is a powerful warlord with super human strength from the land of Sparta in Greece during a time of gods and conflict. Kratos also has a family, a wife and a daughter, whom he loves very much.'),
 Document(metadata={'documentID': '001', 'source': 'ABC XYZ'}, page_content='revealed to be the father of Kratos, kills Kratos to end his menace, to which Kratos swears revenge. Kratos escapes hell and travels back in time by defeating and taking advantage of the Sisters of')]

In [19]:
response = llm.invoke(f"Who is Kratos? You can use the following context: {context}")

print(response.content)


Kratos is described in the provided text as a powerful warlord from Sparta, Greece, with superhuman strength, living in a time of gods and conflict. He has a wife and a daughter whom he loves.

The excerpts also mention a storyline where someone revealed to be Kratos’ father kills Kratos to end his menace, prompting Kratos to swear revenge. He then escapes Hell and travels back in time by defeating the Sisters of Fate. Note that these lines are fragmentary and may not reflect a complete or consistent plot.
